# 🧩 Reto de aplicación — Preprocesamiento y Ensemble
### Bloque 3 · Masterclass Equipo 4
### Dataset: Tips 🍽️

Partimos del dataset **Tips** (propinas en un restaurante) y de un modelo base ya entrenado, para predecir si una comanda fue en **Lunch** o **Dinner** (`time`).

**Tu objetivo en este notebook:**
1. Detectar outliers en otra columna numérica (`tip` o `size`) usando IQR.
2. Crear una feature nueva propia (distinta a `tip_pct`, que ya viene resuelta en la base).
3. Entrenar un Random Forest con esa nueva feature añadida.
4. Comparar el accuracy (y el classification report) contra el modelo base.
5. Entrenar un **XGBoost** con la misma feature nueva.
6. Comparar Random Forest vs. XGBoost — este resultado lo usaremos para el debate del Bloque 4.

Busca los bloques marcados con `# TODO: Implementar aquí` y complétalos. El resto del código ya está resuelto para que puedas partir de una base funcional.


## 0. Setup — instala e importa las librerías necesarias

In [ ]:
# Si estás en Google Colab, descomenta la siguiente línea:
# !pip install xgboost -q

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier


## 1. Punto de partida: el modelo base

Esta parte ya está resuelta — carga el dataset, trata outliers en `total_bill`, crea la feature
`tip_pct` (el % que representa la propina sobre la cuenta) y entrena un Random Forest para predecir
`time` (Lunch/Dinner). Ejecútala tal cual para tener un modelo base con el que comparar después.


In [ ]:
df = sns.load_dataset('tips')
df.head()


In [ ]:
df.isnull().sum()


In [ ]:
# Outliers en total_bill (tratamiento base, ya resuelto)
Q1 = df['total_bill'].quantile(0.25)
Q3 = df['total_bill'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df['total_bill'] = df['total_bill'].clip(lower_bound, upper_bound)

# Feature engineering base: % de propina sobre la cuenta
df['tip_pct'] = df['tip'] / df['total_bill']

df.head()


In [ ]:
num_cols_base = ['total_bill', 'tip', 'size', 'tip_pct']
cat_cols = ['sex', 'smoker', 'day']

X_base = df[num_cols_base + cat_cols]
y = df['time']

X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor_base = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols_base),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

rf_base_pipeline = Pipeline([
    ('preprocessor', preprocessor_base),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

rf_base_pipeline.fit(X_train, y_train)
base_preds = rf_base_pipeline.predict(X_test)
base_accuracy = accuracy_score(y_test, base_preds)

print(f"✅ Accuracy del modelo BASE (Random Forest): {base_accuracy:.3f}")
print(classification_report(y_test, base_preds))


---
## 🧩 A partir de aquí empieza el reto

Guarda siempre tus resultados en las variables indicadas en cada celda — las usaremos más adelante para comparar.

Fíjate en el `classification_report` de arriba: seguramente `Lunch` (la clase minoritaria) tiene peor
precision/recall que `Dinner`. Es un buen ejemplo real de por qué el accuracy solo no cuenta toda la historia.


### Paso 1 — Outliers en otra columna (IQR)

Elige **una** columna: `tip` o `size`.

Pasos a seguir:
1. Calcula `Q1`, `Q3` e `IQR` de la columna elegida.
2. Calcula `lower_bound` y `upper_bound`.
3. Cuenta cuántos outliers hay (imprímelo).
4. Aplica `.clip()` sobre esa columna en `df` para tratarlos.


In [ ]:
outlier_col = 'tip'  # o 'size' — elige uno

# TODO: Implementar aquí
# Q1 = ...
# Q3 = ...
# IQR = ...
# lower_bound = ...
# upper_bound = ...
# outliers_detectados = ...
# print(f"Outliers detectados en {outlier_col}: {len(outliers_detectados)}")
# df[outlier_col] = df[outlier_col].clip(lower_bound, upper_bound)


### Paso 2 — Crea tu propia feature nueva

Distinta a `tip_pct`. Algunas ideas (puedes usar otra si se te ocurre algo mejor):
- `bill_per_person` — dividir `total_bill` entre `size` (cuánto pagó cada comensal de media).
- Un `binning` de `total_bill` en categorías (cuenta baja/media/alta).
- Marcar si el grupo es grande o pequeño (`size` > 2, por ejemplo) como variable binaria.

**Importante:** guarda la columna nueva en `df` con el nombre que tú quieras, y anótalo en `mi_feature_nueva` (como texto) — lo usaremos en el siguiente paso.


In [ ]:
# TODO: Implementar aquí
# df['mi_feature'] = ...
# mi_feature_nueva = 'mi_feature'  # <- pon aquí el nombre exacto de tu columna nueva


### Paso 3 — Entrena un Random Forest con tu nueva feature

Pasos a seguir:
1. Crea una lista `num_cols_reto` = `num_cols_base` + tu nueva columna (`mi_feature_nueva`).
2. Construye un nuevo `ColumnTransformer` (`preprocessor_reto`) igual que el base, pero usando `num_cols_reto`.
3. Crea un nuevo pipeline (`rf_reto_pipeline`) con ese preprocesador y un `RandomForestClassifier` (usa los mismos hiperparámetros que el modelo base para que la comparación sea justa).
4. Entrena con `X_train_reto` / `y_train_reto` — recuerda reconstruir el split porque `X` ahora tiene una columna más.

💡 Pista: tendrás que reconstruir `X` a partir de `df` porque ahora tiene una columna más.


In [ ]:
# TODO: Implementar aquí
# num_cols_reto = num_cols_base + [mi_feature_nueva]
# X_reto = df[num_cols_reto + cat_cols]
# X_train_reto, X_test_reto, y_train_reto, y_test_reto = train_test_split(
#     X_reto, y, test_size=0.2, random_state=42, stratify=y
# )
# preprocessor_reto = ColumnTransformer([...])
# rf_reto_pipeline = Pipeline([...])
# rf_reto_pipeline.fit(X_train_reto, y_train_reto)


### Paso 4 — Compara el Random Forest reto contra el modelo base

Guarda el resultado en `reto_accuracy` e imprime una comparación clara contra `base_accuracy`.


In [ ]:
# TODO: Implementar aquí
# reto_preds = rf_reto_pipeline.predict(X_test_reto)
# reto_accuracy = accuracy_score(y_test_reto, reto_preds)

# print(f"Accuracy modelo BASE (RF):  {base_accuracy:.3f}")
# print(f"Accuracy modelo RETO (RF):  {reto_accuracy:.3f}")
# diferencia = reto_accuracy - base_accuracy
# print(f"Diferencia: {diferencia:+.3f}")
# print(classification_report(y_test_reto, reto_preds))


### Paso 5 — Entrena un XGBoost con la misma feature nueva

XGBoost necesita el target como números, no texto, así que usamos `LabelEncoder` solo para `y`.

Pasos a seguir:
1. Crea un `LabelEncoder`, ajústalo y transforma `y_train_reto` → `y_train_reto_enc`; transforma también `y_test_reto` → `y_test_reto_enc`.
2. Crea un preprocesador nuevo con `clone(preprocessor_reto)` (para no reutilizar uno ya entrenado).
3. Crea un pipeline `xgb_pipeline` con ese preprocesador y un `XGBClassifier` (usa parámetros similares a los que viste en el live coding: `n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss'`).
4. Entrena con `X_train_reto` / `y_train_reto_enc`.


In [ ]:
# TODO: Implementar aquí
# le = LabelEncoder()
# y_train_reto_enc = le.fit_transform(y_train_reto)
# y_test_reto_enc = le.transform(y_test_reto)

# preprocessor_xgb = clone(preprocessor_reto)
# xgb_pipeline = Pipeline([...])
# xgb_pipeline.fit(X_train_reto, y_train_reto_enc)


### Paso 6 — Compara Random Forest vs. XGBoost

Guarda el resultado en `xgb_accuracy` y compáralo con `reto_accuracy` (Random Forest) y `base_accuracy`.
Este resultado es el que llevaremos al debate del Bloque 4.


In [ ]:
# TODO: Implementar aquí
# xgb_preds = xgb_pipeline.predict(X_test_reto)
# xgb_accuracy = accuracy_score(y_test_reto_enc, xgb_preds)

# print(f"Accuracy RF base:      {base_accuracy:.3f}")
# print(f"Accuracy RF reto:      {reto_accuracy:.3f}")
# print(f"Accuracy XGBoost reto: {xgb_accuracy:.3f}")
# print(classification_report(y_test_reto_enc, xgb_preds, target_names=le.classes_))


---
## 💬 Para el debate (Bloque 4)

- ¿Qué columna elegiste para los outliers y cuántos detectaste?
- ¿Qué feature nueva creaste? ¿Por qué pensaste que podía ayudar a distinguir Lunch de Dinner?
- **¿Random Forest o XGBoost dio mejor accuracy en tu caso? ¿Coincide con lo que haya pasado en otras parejas?**
- Con un dataset tan pequeño (244 filas), ¿tiene sentido esperar que XGBoost gane siempre a Random Forest? ¿Por qué sí o por qué no?
- ¿Qué le pasó a la clase minoritaria (`Lunch`) en el classification report de cada modelo?
- Pensando en coste computacional: ¿cuál de los dos creéis que tardaría más en entrenar con un dataset mucho más grande?
